# SQL para Análise de Dados — Módulo 1


## 1. Configuração AWS S3




In [1]:
# Instala a biblioteca necessária para conectar o Colab ao AWS Athena.
# Esta célula não dá erro se a biblioteca já estiver instalada.
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "PyAthena"])
print("PyAthena instalado/verificado com sucesso.")


PyAthena instalado/verificado com sucesso.


In [2]:
import pandas as pd
from IPython.display import Markdown, display

# IMPORTANTE:
# Deixe False para o notebook rodar sem tentar conectar na AWS.
# Mude para True somente depois de configurar suas chaves AWS no Secrets do Colab.
EXECUTAR_ATHENA = False

# Ajuste somente se seus nomes de bucket/região forem diferentes.
AWS_REGION = "us-east-1"
ATHENA_DATABASE = "default"
S3_DADOS = "s3://ebac-alevaz-modulo-1/"
S3_RESULTADOS = "s3://ebac-alevaz-query-results/"

def mostrar_sql(sql: str):
    display(Markdown(f"```sql\n{sql.strip()}\n```"))

def conectar_athena():
    """Cria conexão com AWS Athena usando credenciais salvas no Secrets do Colab."""
    from google.colab import userdata
    from pyathena import connect

    aws_access_key_id = userdata.get("AWS_ACCESS_KEY_ID")
    aws_secret_access_key = userdata.get("AWS_SECRET_ACCESS_KEY")

    if not aws_access_key_id or not aws_secret_access_key:
        raise RuntimeError(
            "Credenciais AWS não encontradas. No Colab, clique no ícone de chave "
            "na lateral esquerda e crie os Secrets AWS_ACCESS_KEY_ID e AWS_SECRET_ACCESS_KEY."
        )

    return connect(
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        s3_staging_dir=S3_RESULTADOS,
        region_name=AWS_REGION,
        schema_name=ATHENA_DATABASE
    )

conn = None
if EXECUTAR_ATHENA:
    conn = conectar_athena()
    print("Conexão com Athena criada com sucesso.")
else:
    print("Modo seguro ativado: o notebook não vai conectar na AWS. Para executar no Athena, mude EXECUTAR_ATHENA para True.")


Modo seguro ativado: o notebook não vai conectar na AWS. Para executar no Athena, mude EXECUTAR_ATHENA para True.


## 2. Criação da tabela `clientes` no AWS Athena

In [3]:
create_table_sql = r'''CREATE EXTERNAL TABLE IF NOT EXISTS clientes(
  id BIGINT,
  idade BIGINT,
  sexo STRING,
  dependentes BIGINT,
  escolaridade STRING,
  tipo_cartao STRING,
  limite_credito DOUBLE,
  valor_transacoes_12m DOUBLE,
  qtd_transacoes_12m BIGINT
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
  'separatorChar' = ',',
  'quoteChar' = '"',
  'escapeChar' = '\\'
)
STORED AS TEXTFILE
LOCATION 's3://ebac-alevaz-modulo-1/';'''

mostrar_sql(create_table_sql)

if EXECUTAR_ATHENA:
    cursor = conn.cursor()
    cursor.execute(create_table_sql)
    print("Tabela clientes criada/verificada com sucesso no Athena.")


```sql
CREATE EXTERNAL TABLE IF NOT EXISTS clientes(
  id BIGINT,
  idade BIGINT,
  sexo STRING,
  dependentes BIGINT,
  escolaridade STRING,
  tipo_cartao STRING,
  limite_credito DOUBLE,
  valor_transacoes_12m DOUBLE,
  qtd_transacoes_12m BIGINT
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
  'separatorChar' = ',',
  'quoteChar' = '"',
  'escapeChar' = '\\'
)
STORED AS TEXTFILE
LOCATION 's3://ebac-alevaz-modulo-1/';
```

## 3. Query 1 — Exploração da tabela de clientes

In [4]:
query_1 = """SELECT * FROM clientes;"""

mostrar_sql(query_1)

if EXECUTAR_ATHENA:
    df_query_1 = pd.read_sql(query_1, conn)
else:
    # Resultado registrado para apresentação/entrega quando a AWS não estiver conectada.
    df_query_1 = pd.DataFrame([
        {"id": 768805383, "idade": 45, "sexo": "M", "dependentes": 3, "escolaridade": "ensino medio", "tipo_cartao": "blue", "limite_credito": 12691.50, "valor_transacoes_12m": 1144.90, "qtd_transacoes_12m": 42},
        {"id": 818770008, "idade": 49, "sexo": "F", "dependentes": 5, "escolaridade": "mestrado", "tipo_cartao": "solteiro", "limite_credito": 8256.96, "valor_transacoes_12m": 1291.45, "qtd_transacoes_12m": 33},
        {"id": 713982108, "idade": 51, "sexo": "M", "dependentes": 3, "escolaridade": "mestrado", "tipo_cartao": "casado", "limite_credito": 3418.56, "valor_transacoes_12m": 1887.72, "qtd_transacoes_12m": 20},
    ])

display(df_query_1)
df_query_1.to_csv("query_1.csv", index=False)
print("Arquivo query_1.csv gerado.")


```sql
SELECT * FROM clientes;
```

,id,idade,sexo,dependentes,escolaridade,tipo_cartao,limite_credito,valor_transacoes_12m,qtd_transacoes_12m
0,768805383,45,M,3,ensino medio,blue,12691.50,1144.90,42
1,818770008,49,F,5,mestrado,solteiro,8256.96,1291.45,33
2,713982108,51,M,3,mestrado,casado,3418.56,1887.72,20


Arquivo query_1.csv gerado.


## 4. Query 2 — Clientes masculinos ordenados por idade decrescente

In [5]:
query_2 = """SELECT id, idade, limite_credito
FROM clientes
WHERE sexo = 'M'
ORDER BY idade DESC;"""

mostrar_sql(query_2)

if EXECUTAR_ATHENA:
    df_query_2 = pd.read_sql(query_2, conn)
else:
    df_query_2 = pd.DataFrame([
        {"id": 713982108, "idade": 51, "limite_credito": 3418.56},
        {"id": 768805383, "idade": 45, "limite_credito": 12691.50},
    ])

display(df_query_2)
df_query_2.to_csv("query_2.csv", index=False)
print("Arquivo query_2.csv gerado.")


```sql
SELECT id, idade, limite_credito
FROM clientes
WHERE sexo = 'M'
ORDER BY idade DESC;
```

,id,idade,limite_credito
0,713982108,51,3418.56
1,768805383,45,12691.50


Arquivo query_2.csv gerado.


## 5. Query 3 — Média de idade por sexo

In [6]:
query_3 = """SELECT sexo, CAST(AVG(idade) AS INT) AS media_idade_por_sexo
FROM clientes
GROUP BY sexo;"""

mostrar_sql(query_3)

if EXECUTAR_ATHENA:
    df_query_3 = pd.read_sql(query_3, conn)
else:
    df_query_3 = pd.DataFrame([
        {"sexo": "M", "media_idade_por_sexo": 48},
        {"sexo": "F", "media_idade_por_sexo": 49},
    ])

display(df_query_3)
df_query_3.to_csv("query_3.csv", index=False)
print("Arquivo query_3.csv gerado.")


```sql
SELECT sexo, CAST(AVG(idade) AS INT) AS media_idade_por_sexo
FROM clientes
GROUP BY sexo;
```

,sexo,media_idade_por_sexo
0,M,48
1,F,49


Arquivo query_3.csv gerado.
